# Solar Energy Analysis
Python-based analysis of a residential property in Grays, Essex, UK, examining its 2024 electricity usage following the installation of a 9.84 kWp solar PV system and 13 kWh battery storage system, including solar generation, grid consumption, electricity costs, export earnings, savings, and net energy usage.

## Dataset
This analysis uses two monthly datasets covering January to December 2024. The MyEnergi dataset contains the amount of electricity consumed by the property that was generated by its solar PV system, measured in kWh. The Octopus Energy dataset contains monthly grid electricity consumption, electricity export, average import unit rates, and average export unit rates.

Together, these datasets provide the information required to analyse solar generation, grid dependency, export earnings, electricity costs, and overall energy savings throughout 2024.

In [1]:
import pandas as pd

# Load datasets
myenergi_df = pd.read_csv("dataset/myenergi_data.csv")
octopus_df = pd.read_csv("dataset/octopus_energy_data.csv")

In [2]:
from IPython.display import display, Markdown

display(Markdown("### MyEnergi Data"))
display(myenergi_df)

display(Markdown("### Octopus Energy Data"))
display(octopus_df)

### MyEnergi Data

,month,solar_electricity_consumption
0,January,37.6kWh
1,February,279.2kWh
2,March,556.3kWh
3,April,653.7kWh
4,May,480.8kWh
5,June,503kWh
6,July,477.8kWh
7,August,460.7kWh
8,September,459.9kWh
9,October,383.3kWh


### Octopus Energy Data

,month,grid_consumption,avg_consumption_unit_rate,exported,avg_export_unit_rate
0,January,755.795kWh,27.82p/kWh,1.012kWh,14.86p/kWh
1,February,499.738kWh,27.82p/kWh,3.487kWh,15p/kWh
2,March,269.413kWh,27.82p/kWh,49.795kWh,15p/kWh
3,April,119.154kWh,24.05p/kWh,229.587kWh,15p/kWh
4,May,31.403kWh,24.05p/kWh,563.598kWh,15p/kWh
5,June,22.679kWh,24.05p/kWh,633.146kWh,15p/kWh
6,July,6.956kWh,21.98p/kWh,652.86kWh,15p/kWh
7,August,12.112kWh,21.98p/kWh,630.079kWh,15p/kWh
8,September,160.123kWh,21.98p/kWh,196.093kWh,15p/kWh
9,October,652.971kWh,24.01p/kWh,8.01kWh,15p/kWh


## Data Cleaning

In [3]:
# Merging dataframes on the column 'month'
merged_df = pd.merge(
    myenergi_df,
    octopus_df,
    on="month",
    how="inner",
    validate="one_to_one"
)

In [4]:
display(merged_df)

,month,solar_electricity_consumption,grid_consumption,avg_consumption_unit_rate,exported,avg_export_unit_rate
0,January,37.6kWh,755.795kWh,27.82p/kWh,1.012kWh,14.86p/kWh
1,February,279.2kWh,499.738kWh,27.82p/kWh,3.487kWh,15p/kWh
2,March,556.3kWh,269.413kWh,27.82p/kWh,49.795kWh,15p/kWh
3,April,653.7kWh,119.154kWh,24.05p/kWh,229.587kWh,15p/kWh
4,May,480.8kWh,31.403kWh,24.05p/kWh,563.598kWh,15p/kWh
5,June,503kWh,22.679kWh,24.05p/kWh,633.146kWh,15p/kWh
6,July,477.8kWh,6.956kWh,21.98p/kWh,652.86kWh,15p/kWh
7,August,460.7kWh,12.112kWh,21.98p/kWh,630.079kWh,15p/kWh
8,September,459.9kWh,160.123kWh,21.98p/kWh,196.093kWh,15p/kWh
9,October,383.3kWh,652.971kWh,24.01p/kWh,8.01kWh,15p/kWh


In [5]:
# Create a copy before cleaning
energy_analysis_df = merged_df.copy()

# Columns that should be numeric
numeric_columns = [
    "solar_electricity_consumption",
    "grid_consumption",
    "avg_consumption_unit_rate",
    "exported",
    "avg_export_unit_rate"
]

# Remove units and convert to numeric
for column in numeric_columns:
    energy_analysis_df[column] = (
        energy_analysis_df[column]
        .astype(str)
        .str.extract(r"(-?\d+(?:\.\d+)?)")[0]
        .pipe(pd.to_numeric, errors="coerce")
    )

# Rename columns to show units
energy_analysis_df = energy_analysis_df.rename(columns={
    "solar_electricity_consumption": "solar_electricity_consumption_kWh",
    "grid_consumption": "grid_consumption_kWh",
    "avg_consumption_unit_rate": "avg_consumption_unit_rate_p_per_kWh",
    "exported": "exported_kWh",
    "avg_export_unit_rate": "avg_export_unit_rate_p_per_kWh"
})

In [6]:
display(energy_analysis_df)
energy_analysis_df.info()

,month,solar_electricity_consumption_kWh,grid_consumption_kWh,avg_consumption_unit_rate_p_per_kWh,exported_kWh,avg_export_unit_rate_p_per_kWh
0,January,37.6,755.795,27.82,1.012,14.86
1,February,279.2,499.738,27.82,3.487,15.00
2,March,556.3,269.413,27.82,49.795,15.00
3,April,653.7,119.154,24.05,229.587,15.00
4,May,480.8,31.403,24.05,563.598,15.00
5,June,503.0,22.679,24.05,633.146,15.00
6,July,477.8,6.956,21.98,652.860,15.00
7,August,460.7,12.112,21.98,630.079,15.00
8,September,459.9,160.123,21.98,196.093,15.00
9,October,383.3,652.971,24.01,8.010,15.00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   month                                12 non-null     object 
 1   solar_electricity_consumption_kWh    12 non-null     float64
 2   grid_consumption_kWh                 12 non-null     float64
 3   avg_consumption_unit_rate_p_per_kWh  12 non-null     float64
 4   exported_kWh                         12 non-null     float64
 5   avg_export_unit_rate_p_per_kWh       12 non-null     float64
dtypes: float64(5), object(1)
memory usage: 708.0+ bytes


## Data Analysis

In [7]:
# Add net consumption column
energy_analysis_df["total_consumption_kWh"] = (
    energy_analysis_df["grid_consumption_kWh"]
    + energy_analysis_df["solar_electricity_consumption_kWh"]
)

In [8]:
# Insert actual bill amount column
energy_analysis_df["actual_bill_gbp"] = (
    energy_analysis_df["grid_consumption_kWh"] * (energy_analysis_df["avg_consumption_unit_rate_p_per_kWh"] / 100)
)

In [9]:
# Insert theoretical bill amount column
energy_analysis_df["theoretical_bill_gbp"] = (
    energy_analysis_df["total_consumption_kWh"] * (energy_analysis_df["avg_consumption_unit_rate_p_per_kWh"] / 100)
)

In [10]:
# Insert savings column
energy_analysis_df["savings_gbp"] = (
    energy_analysis_df["theoretical_bill_gbp"] - energy_analysis_df["actual_bill_gbp"]
)

In [ ]:
# Insert export earnings column
energy_analysis_df["export_earnings_gbp"] = (
    energy_analysis_df["exported_kWh"]
    * (energy_analysis_df["avg_export_unit_rate_p_per_kWh"] / 100)
)

In [13]:
display(energy_analysis_df)

,month,solar_electricity_consumption_kWh,grid_consumption_kWh,avg_consumption_unit_rate_p_per_kWh,exported_kWh,avg_export_unit_rate_p_per_kWh,total_consumption_kWh,actual_bill_gbp,theoretical_bill_gbp,savings_gbp,export_earnings_gbp
0,January,37.6,755.795,27.82,1.012,14.86,793.395,210.262169,220.722489,10.46032,0.150383
1,February,279.2,499.738,27.82,3.487,15.00,778.938,139.027112,216.700552,77.67344,0.523050
2,March,556.3,269.413,27.82,49.795,15.00,825.713,74.950697,229.713357,154.76266,7.469250
3,April,653.7,119.154,24.05,229.587,15.00,772.854,28.656537,185.871387,157.21485,34.438050
4,May,480.8,31.403,24.05,563.598,15.00,512.203,7.552422,123.184821,115.63240,84.539700
5,June,503.0,22.679,24.05,633.146,15.00,525.679,5.454300,126.425800,120.97150,94.971900
6,July,477.8,6.956,21.98,652.860,15.00,484.756,1.528929,106.549369,105.02044,97.929000
7,August,460.7,12.112,21.98,630.079,15.00,472.812,2.662218,103.924078,101.26186,94.511850
8,September,459.9,160.123,21.98,196.093,15.00,620.023,35.195035,136.281055,101.08602,29.413950
9,October,383.3,652.971,24.01,8.010,15.00,1036.271,156.778337,248.808667,92.03033,1.201500
